<a href="https://colab.research.google.com/github/BrandosQuest/CyberMediaProject/blob/master/SpectrogramComparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install musdb mir_eval soundfile

In [16]:
import librosa
import matplotlib.pyplot as plt
import numpy as np
import glob
import IPython.display as ipd
import musdb
import IPython.display as ipd
import mir_eval
import soundfile as sf

In [14]:
def add_white_noise(signal, snr_db):
  sig_power = np.mean(signal**2)
  snr = 10**(snr_db/10)
  noise_power = sig_power/snr

  noise = np.random.normal(0, np.sqrt(noise_power), signal.shape)

  return signal + noise

In [18]:
!mkdir songs

In [19]:
# This downloads a small 7-second sample dataset for testing
mus = musdb.DB(download=True)
track = mus[0]
sr = track.rate
audio = track.audio
snr_target = 5
audio_noisy = add_white_noise(audio, snr_target)
sf.write('./songs/testSample_noisy.wav', audio_noisy, sr)
audio = np.mean(audio, axis=1)

In [ ]:
!mkdemucs_outputs = glob.glob("output/*.wav")
HPS_outputs = glob.glob("output/hps/*.wav")

demucs_stems = {}
hps_stems = {}

for file_path in demucs_outputs:
    stem_name = file_path.split('_')[-1].replace('.wav', '')
    demucs_stems[stem_name] = librosa.load(file_path, sr=sr)

for file_path in HPS_outputs:
    stem_name = file_path.split('\\')[-1].replace('.wav', '')
    hps_stems[stem_name] = librosa.load(file_path, sr=sr)

#demucs_stems

In [ ]:
stft = librosa.stft(audio)

stft_stems_demucs = {name: librosa.stft(stem_audio[0]) for name, stem_audio in demucs_stems.items()}
stft_stems_hps = {name: librosa.stft(stem_audio[0]) for name, stem_audio in hps_stems.items()}

stftList = [stft] + list(stft_stems_demucs.values()) + list(stft_stems_hps.values())

#stft_stems

In [ ]:
dbList = []
for stft_data in stftList:
    dbList.append(librosa.amplitude_to_db(numpy.abs(stft_data), ref=numpy.max))

namesList = ["Original Audio"] + list(stft_stems_demucs.keys()) + list(stft_stems_hps.keys())

#namesList

In [ ]:
fig, ax = plt.subplots(nrows=len(namesList), ncols=1, figsize=(30, 30), sharex=True, sharey=True, layout='constrained')
specshow_kwargs = dict(sr=sr, x_axis='time', y_axis='log', cmap='magma')

for i, dbStft in enumerate(dbList):
    img = librosa.display.specshow(dbStft, ax=ax[i], **specshow_kwargs)
    fig.colorbar(img, ax=ax[i], format='%+2.0f dB')
    ax[i].set_title(f'Spectrogram {namesList[i]}')

fig.savefig('spectrograms/spectrograms_compared.png', dpi=150, bbox_inches='tight')


#plt.figure(figsize=(10, 6))
#librosa.display.specshow(stft_db, sr=sr, x_axis='time', y_axis='hz')
#plt.colorbar(format='%+2.0f dB')#capire per quale motivo dobbiamo utilizare i decibel
#plt.title('Spectrogram')
#plt.show()

In [ ]:
mus = musdb.DB(download=True)
track = mus[0]

print(f"Original Audio: {track.name}")
display(ipd.Audio(track.audio.T, rate=track.rate))

#print("Isolated Vocals:")
#display(ipd.Audio(track.targets['vocals'].audio.T, rate=track.rate))
#print("Isolated Drums:")
#display(ipd.Audio(track.targets['drums'].audio.T, rate=track.rate))
#print("Isolated other:")
#display(ipd.Audio(track.targets['other'].audio.T, rate=track.rate))
#print("Isolated bass:")
#display(ipd.Audio(track.targets['bass'].audio.T, rate=track.rate))


true_vocals = numpy.mean(track.targets['vocals'].audio, axis=1)
true_drums = numpy.mean(track.targets['drums'].audio, axis=1)

est_vocals, _ = demucs_stems["vocals"]
est_drums, _ = demucs_stems["drums"]

target_len = len(true_vocals)
est_vocals = librosa.util.fix_length(est_vocals, size=target_len)
est_drums = librosa.util.fix_length(est_drums, size=target_len)

reference_sources = numpy.vstack([true_vocals, true_drums])
estimated_sources = numpy.vstack([est_vocals, est_drums])

sdr, sir, sar, perm = mir_eval.separation.bss_eval_sources(
    reference_sources,
    estimated_sources,
    compute_permutation=False
)

print(f"Risultati Voce     -> SDR: {sdr[0]:.2f} dB, SIR: {sir[0]:.2f} dB, SAR: {sar[0]:.2f} dB")
print(f"Risultati Batteria -> SDR: {sdr[1]:.2f} dB, SIR: {sir[1]:.2f} dB, SAR: {sar[1]:.2f} dB")
"""The Metrics Explained
SDR (Signal-to-Distortion Ratio): This is the "Gold Standard" metric. It measures the overall quality of the separated track. Higher is always better.

SIR (Signal-to-Interference Ratio): This measures bleed or leakage. A high SIR means the algorithm successfully blocked out the other instruments.

SAR (Signal-to-Artifacts Ratio): This measures digital artifacts. A low SAR means the track probably sounds "watery," "robotic," or hollow (like a low-quality MP3), even if the other instruments are gone."""

In [ ]:
import matplotlib.pyplot as plt
import gc

plt.close('all') # Closes all hidden figure windows
gc.collect()     # Forces Python to run garbage collection and free RAM